# Notebook 16 — Pseudo-Labeled LGBM

**Goal:** Re-train the NB12 LGBM on `train + high-confidence test pseudo-labels` to see if adding ~70k
confidently-predicted test rows improves generalization. The test predictions are drawn from the v004
ensemble (rank avg MLP NB15 + LGBM NB12) — the strongest signal we have for the test set.

**Why pseudo-labeling can help:**  
The test set is drawn from the same underlying distribution as train. High-confidence predictions
(p < 0.02 or p > 0.85) are very likely correct, and adding them to training can expose the model to
test-set-specific patterns (race-year combinations, driver identities) not fully represented in train.

**Pseudo-label source:** Rank average of `lgbm_tuned_pred` (NB12) + `mlp_pred` (NB15) — v004 equivalent.  
**Thresholds:** p < 0.02 → label=0; p > 0.85 → label=1. (~37% of test rows get a label.)  
**LGBM params:** Exact NB12 re-tuned params — no re-tuning here.  

**Validation protocol:** GroupKFold(5) on **train rows only**. Pseudo-labeled test rows are appended
to each fold's training set — never to the validation set. OOF AUC is computed on train rows only.

**Gate:** CV gain ≥ +0.001 vs NB12 baseline (0.9024). If gain < +0.001, save the notebook but do NOT
use this model in NB20 — fall back to NB12 LGBM.

**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`,
`test_predictions_nb12.parquet` (lgbm_tuned_pred), `test_predictions_nb15.parquet` (mlp_pred)  
**Outputs:** `oof_predictions_nb16.parquet`, `test_predictions_nb16.parquet`, `models/nb16_summary.pkl`

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print(f'LightGBM version: {lgb.__version__}')

LightGBM version: 4.6.0


In [2]:
# Robust project root detection — works from workspace root, notebooks/, or Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root  = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

if not processed_dir.exists():
    processed_dir = Path('/kaggle/working')

print(f'Project root : {project_root}')
print(f'Processed dir: {processed_dir}')

train = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test  = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds = pd.read_parquet(processed_dir / 'fold_assignments.parquet')

train = train.merge(folds, on='id', how='left')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Class balance: {train["PitNextLap"].mean():.3f} positive rate')

Project root : c:\Repos\predict-f1-pit-stops
Processed dir: c:\Repos\predict-f1-pit-stops\data\processed
Train: (439140, 53)  |  Test: (188165, 51)
Class balance: 0.199 positive rate


## Construct Pseudo-Labels

Source: rank average of LGBM NB12 + MLP NB15 test predictions (equivalent to the v004 submission).
Using rank avg rather than raw preds avoids scale mismatch between LGBM and MLP probability outputs.

In [3]:
test_lgbm = pd.read_parquet(processed_dir / 'test_predictions_nb12.parquet')[['id', 'lgbm_tuned_pred']]
test_mlp  = pd.read_parquet(processed_dir / 'test_predictions_nb15.parquet')[['id', 'mlp_pred']]

# Merge and verify alignment
test_preds = test_lgbm.merge(test_mlp, on='id', how='inner')
assert len(test_preds) == len(test), f'Alignment mismatch: {len(test_preds)} vs {len(test)}'

# Rank-average (same as v004 submission logic)
r_lgbm = rankdata(test_preds['lgbm_tuned_pred'].values) / len(test_preds)
r_mlp  = rankdata(test_preds['mlp_pred'].values)        / len(test_preds)
test_preds['v004_rank_avg'] = (r_lgbm + r_mlp) / 2.0

# Apply thresholds
PL_THRESH_NEG = 0.02   # p < 0.02 → pseudo-label = 0
PL_THRESH_POS = 0.85   # p > 0.85 → pseudo-label = 1

p = test_preds['v004_rank_avg'].values
mask_neg = p < PL_THRESH_NEG
mask_pos = p > PL_THRESH_POS
mask_pl  = mask_neg | mask_pos

test_preds['pseudo_label'] = np.where(mask_pos, 1, np.where(mask_neg, 0, -1))

print(f'Pseudo-label thresholds: p < {PL_THRESH_NEG} → 0  |  p > {PL_THRESH_POS} → 1')
print(f'  label=0 : {mask_neg.sum():>6,} rows ({mask_neg.mean()*100:.1f}%)')
print(f'  label=1 : {mask_pos.sum():>6,} rows ({mask_pos.mean()*100:.1f}%)')
print(f'  unlabeled: {(~mask_pl).sum():>6,} rows ({(~mask_pl).mean()*100:.1f}%)')
print(f'  total pseudo-labeled: {mask_pl.sum():,} rows ({mask_pl.mean()*100:.1f}% of test)')
print(f'  pseudo-label positive rate: {mask_pos.sum() / mask_pl.sum():.3f}')

Pseudo-label thresholds: p < 0.02 → 0  |  p > 0.85 → 1
  label=0 :  1,314 rows (0.7%)
  label=1 : 28,171 rows (15.0%)
  unlabeled: 158,680 rows (84.3%)
  total pseudo-labeled: 29,485 rows (15.7% of test)
  pseudo-label positive rate: 0.955


In [4]:
# Build pseudo-labeled test DataFrame — same columns as train (except fold)
pl_ids = test_preds.loc[mask_pl, 'id'].values
pl_labels = test_preds.loc[mask_pl, 'pseudo_label'].values

test_pl = test[test['id'].isin(pl_ids)].copy()
test_pl = test_pl.merge(test_preds[['id', 'pseudo_label']], on='id', how='left')
test_pl['PitNextLap'] = test_pl['pseudo_label'].astype(np.int8)
test_pl = test_pl.drop(columns=['pseudo_label'])

print(f'Pseudo-labeled test rows: {len(test_pl):,}')
print(f'Positive rate in pseudo-labels: {test_pl["PitNextLap"].mean():.3f}')
print(f'Original train positive rate  : {train["PitNextLap"].mean():.3f}')

Pseudo-labeled test rows: 29,485
Positive rate in pseudo-labels: 0.955
Original train positive rate  : 0.199


## Feature Set + Target Encodings

Identical 38-feature set from NB11/NB12. Target encodings computed fold-aware inside each CV fold
(from train-fold rows only — applied to val rows AND pseudo-labeled test rows).

In [5]:
BASE_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
TARGET_ENC = ['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length']
TIER14_FEATS = [
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    'prime_pit_window', 'prime_window_x_compound',
    'abs_position_change', 'pos_change_rolling_std_3',
    'PitStop_lag1', 'laps_to_driver_end',
]
FEAT_COLS = BASE_FEATURES + TARGET_ENC + TIER14_FEATS  # 38 total
print(f'Feature count: {len(FEAT_COLS)}')


def apply_target_encodings(train_df: pd.DataFrame, val_df: pd.DataFrame) -> pd.DataFrame:
    """Fold-aware target encodings. Computed from train_df, applied to val_df."""
    global_mean  = train_df['PitNextLap'].mean()
    race_map     = train_df.groupby('Race')['PitNextLap'].mean()
    driver_map   = train_df.groupby('Driver')['PitNextLap'].mean()
    stint_map    = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max().reset_index().groupby('Driver')['TyreLife'].mean()
    )
    global_stint = stint_map.mean()

    out = val_df.copy()
    out['Race_target_encoded']     = out['Race'].map(race_map).fillna(global_mean)
    out['Driver_target_encoded']   = out['Driver'].map(driver_map).fillna(global_mean)
    out['Driver_avg_stint_length'] = out['Driver'].map(stint_map).fillna(global_stint)
    return out

print('apply_target_encodings defined.')

Feature count: 38
apply_target_encodings defined.


## 5-Fold GroupKFold CV — LGBM + Pseudo-Labels

For each fold:
1. Compute fold-aware target encodings from `train.iloc[tr_idx]`
2. Apply to val rows and to pseudo-labeled test rows
3. Concatenate `train_enc[tr_idx]` + `test_pl_enc` → training set
4. Train LGBM with NB12 params, early stopping on val (train rows only)
5. Predict OOF on `train.iloc[val_idx]`

In [6]:
# NB12 re-tuned LGBM params (exact)
LGBM_PARAMS = {
    'objective':          'binary',
    'metric':             'auc',
    'learning_rate':       0.022004860219019283,
    'num_leaves':          31,
    'min_child_samples':   12,
    'subsample':           0.9678695939801807,
    'colsample_bytree':    0.4298227704355011,
    'reg_alpha':           17.28167401662238,
    'reg_lambda':          1.8103877761997072e-05,
    'max_bin':             499,
    'min_gain_to_split':   0.3691288933102477,
    'random_state':        SEED,
    'verbose':            -1,
    'n_jobs':             -1,
}
N_ESTIMATORS  = 3000   # ceiling; early stopping will find the right number
EARLY_STOP    = 50     # same as NB12
NB12_BASELINE = 0.9024 # OOF AUC from NB12
GAIN_GATE     = 0.001  # minimum improvement to include in NB20 ensemble

print('LGBM params loaded (NB12 re-tuned).')
print(f'N_estimators ceiling: {N_ESTIMATORS}  |  Early stop: {EARLY_STOP}')
print(f'Gate: OOF AUC ≥ {NB12_BASELINE + GAIN_GATE:.4f} (+{GAIN_GATE})')

LGBM params loaded (NB12 re-tuned).
N_estimators ceiling: 3000  |  Early stop: 50
Gate: OOF AUC ≥ 0.9034 (+0.001)


In [7]:
n_train      = len(train)
oof_preds    = np.zeros(n_train, dtype=np.float64)
test_fold_preds = []
fold_aucs    = []
fold_n_trees = []

print(f'Training 5-fold LGBM + pseudo-labels  (pseudo-labeled rows: {len(test_pl):,})')
print('=' * 72)

for fold_idx in range(5):
    t0 = time.time()
    np.random.seed(SEED + fold_idx)

    tr_idx  = np.where(train['fold'] != fold_idx)[0]
    val_idx = np.where(train['fold'] == fold_idx)[0]

    # Fold-aware target encodings (computed from train fold only)
    train_enc = apply_target_encodings(train.iloc[tr_idx], train.iloc[tr_idx])
    val_enc   = apply_target_encodings(train.iloc[tr_idx], train.iloc[val_idx])
    test_enc  = apply_target_encodings(train.iloc[tr_idx], test)
    test_pl_enc = apply_target_encodings(train.iloc[tr_idx], test_pl)

    # Concatenate: original train fold + pseudo-labeled test rows
    combined_train = pd.concat([train_enc, test_pl_enc], ignore_index=True)
    X_tr = combined_train[FEAT_COLS].values
    y_tr = combined_train['PitNextLap'].values

    X_val = val_enc[FEAT_COLS].values
    y_val = val_enc['PitNextLap'].values

    X_te  = test_enc[FEAT_COLS].values

    ds_tr  = lgb.Dataset(X_tr, label=y_tr, feature_name=FEAT_COLS, free_raw_data=False)
    ds_val = lgb.Dataset(X_val, label=y_val, feature_name=FEAT_COLS, reference=ds_tr, free_raw_data=False)

    callbacks = [
        lgb.early_stopping(EARLY_STOP, verbose=False),
        lgb.log_evaluation(period=200),
    ]

    model = lgb.train(
        LGBM_PARAMS,
        ds_tr,
        num_boost_round=N_ESTIMATORS,
        valid_sets=[ds_val],
        callbacks=callbacks,
    )

    n_trees = model.best_iteration
    fold_n_trees.append(n_trees)

    oof_preds[val_idx] = model.predict(X_val, num_iteration=n_trees)
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_aucs.append(fold_auc)

    test_fold_preds.append(model.predict(X_te, num_iteration=n_trees))

    elapsed = time.time() - t0
    tr_rows = len(combined_train)
    print(f'Fold {fold_idx}: AUC={fold_auc:.4f}  trees={n_trees}  '
          f'train_rows={tr_rows:,} ({len(tr_idx):,}+{len(test_pl):,} pseudo)  ({elapsed/60:.1f} min)')

test_preds_final = np.mean(test_fold_preds, axis=0)
oof_auc = roc_auc_score(train['PitNextLap'], oof_preds)

print()
print('=' * 72)
print(f'OOF AUC (pseudo-labeled LGBM): {oof_auc:.4f}  |  fold std: {np.std(fold_aucs):.4f}')
print(f'NB12 baseline                : {NB12_BASELINE:.4f}')
print(f'Delta                        : {oof_auc - NB12_BASELINE:+.4f}')
print(f'Gate ({NB12_BASELINE + GAIN_GATE:.4f})               : {"PASS" if oof_auc >= NB12_BASELINE + GAIN_GATE else "FAIL"}')
print(f'Fold AUCs : {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Best trees: {fold_n_trees}')

Training 5-fold LGBM + pseudo-labels  (pseudo-labeled rows: 29,485)
[200]	valid_0's auc: 0.904081
[400]	valid_0's auc: 0.910331
[600]	valid_0's auc: 0.912544
[800]	valid_0's auc: 0.913691
[1000]	valid_0's auc: 0.914109
[1200]	valid_0's auc: 0.914531
[1400]	valid_0's auc: 0.914974
[1600]	valid_0's auc: 0.915169
Fold 0: AUC=0.9152  trees=1595  train_rows=380,607 (351,122+29,485 pseudo)  (0.5 min)
[200]	valid_0's auc: 0.901811
[400]	valid_0's auc: 0.910346
[600]	valid_0's auc: 0.913304
[800]	valid_0's auc: 0.914665
[1000]	valid_0's auc: 0.91553
[1200]	valid_0's auc: 0.91611
[1400]	valid_0's auc: 0.916517
Fold 1: AUC=0.9167  trees=1503  train_rows=381,181 (351,696+29,485 pseudo)  (0.5 min)
[200]	valid_0's auc: 0.909049
[400]	valid_0's auc: 0.918865
[600]	valid_0's auc: 0.920712
[800]	valid_0's auc: 0.921345
[1000]	valid_0's auc: 0.921902
Fold 2: AUC=0.9219  trees=993  train_rows=380,598 (351,113+29,485 pseudo)  (0.3 min)
[200]	valid_0's auc: 0.909262
[400]	valid_0's auc: 0.915175
[600]	val

## Correlation Analysis vs MLP NB15

Check Spearman ρ vs MLP NB15 OOF and vs NB12 LGBM OOF. The pseudo-labeled LGBM will naturally
have very high ρ vs NB12 (same architecture, same features) — that's expected and fine. The diversity
vs MLP is what matters for the NB20 ensemble.

In [8]:
mlp_df  = pd.read_parquet(processed_dir / 'oof_predictions_nb15.parquet')
lgbm_df = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')

# Align by id to match oof_preds row order (train is not necessarily id-sorted)
train_s  = train.sort_values('id').reset_index(drop=True)
mlp_df   = mlp_df.sort_values('id').reset_index(drop=True)
lgbm_df  = lgbm_df.sort_values('id').reset_index(drop=True)

id_to_pos = {row_id: pos for pos, row_id in enumerate(train['id'].values)}
oof_sorted = oof_preds[[id_to_pos[i] for i in train_s['id'].values]]

mlp_oof  = mlp_df['mlp_oof'].values
lgbm_oof = lgbm_df['lgbm_tuned_oof'].values
y_true   = train_s['PitNextLap'].values

rho_mlp,  _ = spearmanr(mlp_oof,  oof_sorted)
rho_lgbm, _ = spearmanr(lgbm_oof, oof_sorted)
rho_ml,   _ = spearmanr(mlp_oof,  lgbm_oof)

print('Spearman ρ analysis')
print(f'  NB16 pseudo-LGBM vs MLP NB15  : {rho_mlp:.4f}')
print(f'  NB16 pseudo-LGBM vs LGBM NB12 : {rho_lgbm:.4f}  (expected ~0.99 — same arch)')
print(f'  MLP NB15 vs LGBM NB12         : {rho_ml:.4f}  (reference: 0.791)')

# Rank-avg ensemble preview
def rank_norm(arr):
    return rankdata(arr) / len(arr)

r_nb16  = rank_norm(oof_sorted)
r_mlp   = rank_norm(mlp_oof)
r_lgbm  = rank_norm(lgbm_oof)

auc_nb16_solo = roc_auc_score(y_true, oof_sorted)
auc_nb16_mlp  = roc_auc_score(y_true, (r_nb16 + r_mlp) / 2)
auc_nb12_mlp  = roc_auc_score(y_true, (r_lgbm + r_mlp) / 2)

print()
print('Ensemble preview (rank avg):')
print(f'  NB16 solo          : {auc_nb16_solo:.4f}')
print(f'  NB12 solo (ref)    : 0.9024')
print(f'  Rank avg NB16+MLP  : {auc_nb16_mlp:.4f}  Δ vs NB12+MLP ({auc_nb12_mlp:.4f}): {auc_nb16_mlp - auc_nb12_mlp:+.4f}')
print()
gate_pass = oof_auc >= NB12_BASELINE + GAIN_GATE
print(f'Decision: {"✅ INCLUDE in NB20 super-ensemble (gate passed)" if gate_pass else "❌ EXCLUDE from NB20 — use NB12 LGBM instead (gate failed)"}')

Spearman ρ analysis
  NB16 pseudo-LGBM vs MLP NB15  : 0.8244
  NB16 pseudo-LGBM vs LGBM NB12 : 0.9705  (expected ~0.99 — same arch)
  MLP NB15 vs LGBM NB12         : 0.7910  (reference: 0.791)

Ensemble preview (rank avg):
  NB16 solo          : 0.9153
  NB12 solo (ref)    : 0.9024
  Rank avg NB16+MLP  : 0.9203  Δ vs NB12+MLP (0.9150): +0.0053

Decision: ✅ INCLUDE in NB20 super-ensemble (gate passed)


## Save Outputs

In [9]:
# OOF predictions (row order matches original train)
oof_df = pd.DataFrame({
    'id':          train['id'].values,
    'fold':        train['fold'].values,
    'PitNextLap':  train['PitNextLap'].values,
    'lgbm_pl_oof': oof_preds,
})
oof_path = processed_dir / 'oof_predictions_nb16.parquet'
oof_df.to_parquet(oof_path, index=False)
print(f'Saved OOF : {oof_path}  shape={oof_df.shape}')

# Test predictions
test_df_out = pd.DataFrame({'id': test['id'].values, 'lgbm_pl_pred': test_preds_final})
test_path = processed_dir / 'test_predictions_nb16.parquet'
test_df_out.to_parquet(test_path, index=False)
print(f'Saved test: {test_path}  shape={test_df_out.shape}')

# Summary
summary = {
    'oof_auc':            float(oof_auc),
    'fold_aucs':          [float(a) for a in fold_aucs],
    'fold_std':           float(np.std(fold_aucs)),
    'fold_n_trees':       fold_n_trees,
    'nb12_baseline':      NB12_BASELINE,
    'delta_vs_nb12':      float(oof_auc - NB12_BASELINE),
    'gate_pass':          bool(gate_pass),
    'pl_thresh_neg':      PL_THRESH_NEG,
    'pl_thresh_pos':      PL_THRESH_POS,
    'n_pseudo_labeled':   int(mask_pl.sum()),
    'n_pseudo_neg':       int(mask_neg.sum()),
    'n_pseudo_pos':       int(mask_pos.sum()),
    'spearman_rho_mlp':   float(rho_mlp),
    'spearman_rho_lgbm12':float(rho_lgbm),
    'auc_rank_nb16_mlp':  float(auc_nb16_mlp),
    'auc_rank_nb12_mlp':  float(auc_nb12_mlp),
    'lgbm_params':        LGBM_PARAMS,
}
summary_path = models_dir / 'nb16_summary.pkl'
with open(summary_path, 'wb') as f:
    pickle.dump(summary, f)
print(f'Saved summary: {summary_path}')

print()
print('─' * 72)
print('NB16 RESULTS:')
print(f'  OOF AUC      : {oof_auc:.4f}  (NB12 baseline: {NB12_BASELINE:.4f}  Δ={oof_auc - NB12_BASELINE:+.4f})')
print(f'  Fold AUCs    : {[f"{a:.4f}" for a in fold_aucs]}')
print(f'  Fold std     : {np.std(fold_aucs):.4f}')
print(f'  Gate (≥+0.001): {"PASS" if gate_pass else "FAIL"}')
print(f'  ρ vs MLP     : {rho_mlp:.4f}')
print(f'  Rank avg NB16+MLP: {auc_nb16_mlp:.4f}  (NB12+MLP: {auc_nb12_mlp:.4f}  Δ={auc_nb16_mlp-auc_nb12_mlp:+.4f})')
print()
print(f'  NB20 LGBM source: {"NB16 (pseudo-labeled)" if gate_pass else "NB12 (original, gate failed)"}')

Saved OOF : c:\Repos\predict-f1-pit-stops\data\processed\oof_predictions_nb16.parquet  shape=(439140, 4)
Saved test: c:\Repos\predict-f1-pit-stops\data\processed\test_predictions_nb16.parquet  shape=(188165, 2)
Saved summary: c:\Repos\predict-f1-pit-stops\models\nb16_summary.pkl

────────────────────────────────────────────────────────────────────────
NB16 RESULTS:
  OOF AUC      : 0.9153  (NB12 baseline: 0.9024  Δ=+0.0129)
  Fold AUCs    : ['0.9152', '0.9167', '0.9219', '0.9190', '0.9008']
  Fold std     : 0.0073
  Gate (≥+0.001): PASS
  ρ vs MLP     : 0.8244
  Rank avg NB16+MLP: 0.9203  (NB12+MLP: 0.9150  Δ=+0.0053)

  NB20 LGBM source: NB16 (pseudo-labeled)


In [10]:
# Submit predictions
# Quick v005 submission — rank avg NB16 LGBM-PL + MLP NB15
from scipy.stats import rankdata

test_nb16 = pd.read_parquet(processed_dir / 'test_predictions_nb16.parquet')
test_nb15 = pd.read_parquet(processed_dir / 'test_predictions_nb15.parquet')

sub = test_nb16.merge(test_nb15, on='id')
r_lgbm_pl = rankdata(sub['lgbm_pl_pred'].values) / len(sub)
r_mlp      = rankdata(sub['mlp_pred'].values)     / len(sub)
sub['PitNextLap'] = (r_lgbm_pl + r_mlp) / 2.0

sub_path = project_root / 'submissions' / 'submission_v005_lgbm_pl_mlp.csv'
sub[['id', 'PitNextLap']].to_csv(sub_path, index=False)
print(f'Saved: {sub_path}  shape={sub[["id","PitNextLap"]].shape}')
print(f'Pred range: {sub["PitNextLap"].min():.4f} – {sub["PitNextLap"].max():.4f}')


Saved: c:\Repos\predict-f1-pit-stops\submissions\submission_v005_lgbm_pl_mlp.csv  shape=(188165, 2)
Pred range: 0.0002 – 0.9999
